In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [3]:
df = pd.read_csv("student_performance.csv")

print(df.shape)

df.head()

(10000, 7)


,student_id,weekly_self_study_hours,attendance_percentage,class_participation,total_score,grade,PassFail
0,101128,16.7,91.0,4.3,100.0,A,1
1,5202,11.4,95.3,5.5,88.0,A,1
2,758857,5.4,77.1,8.1,78.1,B,1
3,840329,22.9,87.7,3.5,100.0,A,1
4,6914,23.1,87.4,10.0,96.6,A,1


In [4]:
from sklearn.model_selection import train_test_split

feature_columns = ['weekly_self_study_hours', 'attendance_percentage',
    'class_participation']
X = df[feature_columns]
y = df['PassFail']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

print("Training set size:", X_train.shape)
print("Test set size:", X_test.shape)
print()
print("Pass rate in training set:", y_train.mean().round(3))
print("Pass rate in test set:", y_test.mean().round(3))

Training set size: (8000, 3)
Test set size: (2000, 3)

Pass rate in training set: 0.751
Pass rate in test set: 0.746


In [5]:
from sklearn.metrics import accuracy_score
from sklearn.neighbors import KNeighborsClassifier

knn_baseline = KNeighborsClassifier(n_neighbors=5)
knn_baseline.fit(X_train, y_train)
y_pred_baseline = knn_baseline.predict(X_test)
acc_baseline = accuracy_score(y_test, y_pred_baseline)

print(f"Accuracy WITHOUT scaling (K=5, Euclidean): {acc_baseline*100:.3f}%")

Accuracy WITHOUT scaling (K=5, Euclidean): 87.150%


In [6]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

knn_scaled = KNeighborsClassifier(n_neighbors=5)
knn_scaled.fit(X_train_scaled, y_train)
y_pred_scaled = knn_scaled.predict(X_test_scaled)
acc_scaled = accuracy_score(y_test, y_pred_scaled)

print(f"Accuracy WITHOUT scaling: {acc_baseline*100:.3f}%")
print(f"Accuracy WITH scaling: {acc_scaled*100:.3f}%")

Accuracy WITHOUT scaling: 87.150%
Accuracy WITH scaling: 87.900%


In [10]:
distance_metrics = {
'Euclidean': {'metric': 'euclidean'},
'Manhattan': {'metric': 'manhattan'},
'Minkowski (p=3)': {'metric': 'minkowski', 'p': 3},
}
metric_results = {}
for name, params in distance_metrics.items():
    model = KNeighborsClassifier(n_neighbors=5, **params)
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    acc = accuracy_score(y_test, preds)
    metric_results [name] = acc
    print(f"{name}: accuracy = {acc*100:.3f}%")


Euclidean: accuracy = 87.150%
Manhattan: accuracy = 87.350%
Minkowski (p=3): accuracy = 87.100%


In [11]:
from sklearn.tree import DecisionTreeClassifier, plot_tree

url = 'https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv'
df = pd.read_csv(url)

In [12]:
print(df.shape)

df.head()

(891, 12)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [13]:
base_df = df[['Survived', 'Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked']].copy()
# Impute missing Age with the overall median (a simple, reasonable default for a baseline model)
base_df['Age'] = base_df['Age'].fillna(base_df['Age'].median())
# Impute missing Embarked with the mode (most frequent port)
base_df['Embarked'] = base_df['Embarked'].fillna(base_df['Embarked'].mode()[0])
#scikit-Learn's DecisionTreeClassifier requires numeric input, so we encode the two
# categorical columns. Sex is binary, so a simple mapping is sufficient.
base_df['Sex'] = base_df['Sex'].map({'male': 0, 'female': 1})
base_df['Embarked'] = base_df['Embarked'].map({'S': 0, 'C': 1, 'Q': 2})
print (base_df.isnull().sum().sum(), "missing values remaining")
base_df.head()


0 missing values remaining


,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,0,3,0,22.0,1,0,7.2500,0
1,1,1,1,38.0,1,0,71.2833,1
2,1,3,1,26.0,0,0,7.9250,0
3,1,1,1,35.0,1,0,53.1000,0
4,0,3,0,35.0,0,0,8.0500,0


In [14]:
X_base = base_df.drop(columns=['Survived'])
y_base = base_df['Survived']
X_train_base, X_test_base, y_train_base, y_test_base = train_test_split(
X_base, y_base, test_size=0.2, random_state=42, stratify=y_base
)
print("Training set:", X_train_base.shape)
print("Test set:", X_test_base.shape)
print(f"Train survival rate: {y_train_base.mean():.3f}")
print(f"Test survival rate: {y_test_base.mean():.3f}")


Training set: (712, 7)
Test set: (179, 7)
Train survival rate: 0.383
Test survival rate: 0.385


In [15]:
clf_base = DecisionTreeClassifier(
criterion='gini',
max_depth=4,
random_state=42)
clf_base.fit(X_train_base, y_train_base)
y_pred_base = clf_base.predict(X_test_base)
acc_base = accuracy_score(y_test_base, y_pred_base)
print(f"Baseline model test accuracy: {acc_base:.4f}")

Baseline model test accuracy: 0.7877


In [ ]:
fig, ax = plt.subplots (figsize=(22, 12))
plot_tree(
clf_base,
feature_names=X_base.columns,
class_names=['Died', 'Survived'],
filled=True,
rounded=True,
fontsize=10,
ax=ax)
plt.title('Baseline Decision Tree (max_depth=4, original features)', fontsize=16)
plt.show()